# 00 — Business Understanding

## Projeto CRISP-DM: Propensão à Recompra (Bakehouse)

**Fonte de dados:** `samples.bakehouse.sales_transactions` (somente leitura)  
**Escrita das tabelas derivadas:** `workspace.ml_bakehouse`

## Definição do problema

Estimar a **probabilidade de um cliente ativo realizar pelo menos uma compra no dia seguinte à data de referência**, para dar suporte a campanhas de retenção e acionamento.

| Item | Definição |
| --- | --- |
| Tipo de problema | Classificação binária |
| Unidade amostral | Cliente × data de referência (snapshot) |
| Métricas de sucesso | AUC-ROC, KS, Lift nos decis superiores, calibração (Brier Score) |
| Variável proibida | `cardNumber` (e qualquer PII) excluída da modelagem |

## ⚠️ Adaptação das janelas ao período disponível

O plano original previa **90 dias de histórico e 30 dias de horizonte**. Porém a base cobre apenas **17 dias (2024-05-01 a 2024-05-17)**, com 3.333 transações de 300 clientes — que compram com alta frequência (gap médio entre compras ≈ **1,3 dia**).

Diagnóstico realizado antes de fechar o desenho:

| Cenário testado | Taxa de evento | Avaliação |
| --- | --- | --- |
| Histórico 90d / target 30d | — | **Impossível:** só há 17 dias de dados |
| Histórico 5d / target 3d | ≈ 92% | Ruim: quase todo cliente "recompra" — target trivial |
| **Histórico 5d / target 1d** | **≈ 50%** (medido: 46–55%) | **Escolhido:** classes balanceadas, target informativo |

**Decisão:**
* Janela histórica (features): **5 dias anteriores à referência**
* Horizonte do target: **1 dia** (recompra no dia seguinte)
* Snapshots diários: **2024-05-06 a 2024-05-15** (as primeiras datas exigem 5 dias de histórico; 05-17 foi descartado como referência por conter dados apenas até 11h48 — dia incompleto)

## Definição matemática do target

$$Y_{i,t} = \begin{cases} 1, & \text{se o cliente } i \text{ realiza compra em } (t,\, t+1\text{d}] \\ 0, & \text{caso contrário} \end{cases}$$

**Cliente ativo no snapshot *t*:** pelo menos uma compra em **[t−5d, t−1d]**.

**Regra anti-leakage:** as variáveis preditoras só podem usar informações **até** *t* **(exclusive)**; a janela futura é exclusiva para construir o target.

In [0]:
# ── Configuração central do projeto (ajuste se necessário) ──
SOURCE  = "samples.bakehouse.sales_transactions"   # fonte original — somente leitura
CATALOG = "workspace"                               # catálogo de escrita
SCHEMA  = "ml_bakehouse"                             # schema do projeto

HIST_WINDOW   = 5             # janela histórica em dias (features)
TARGET_WINDOW = 1             # horizonte do target em dias (recompra no dia seguinte)

REF_START = "2024-05-06"      # 1º snapshot: exige 5 dias de histórico completos
REF_END   = "2024-05-15"      # último snapshot: 1 dia de futuro completo

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Schema de trabalho:", f"{CATALOG}.{SCHEMA}")
print("Snapshots:", REF_START, "→", REF_END, "| Janela hist:", HIST_WINDOW, "d | Target:", TARGET_WINDOW, "d")

## Estrutura do projeto — notebooks e entregáveis

| Notebook | Etapa CRISP-DM | Entregável (tabela) |
| --- | --- | --- |
| `01_populacao_ativa` | Data Understanding | `population_active` |
| `02_feature_store` | Data Preparation | `customer_features` |
| `03_target` | Data Preparation | `population_target` |
| `04_eda_i` | Data Understanding | Relatório exploratório |
| `05_eda_ii` | Data Preparation | `feature_analysis` (IV, PSI, correlações) |
| `06_feature_engineering` | Data Preparation | `train_base`, `validation_base`, `test_base` (com `safra`) |
| `07_sampling` | Data Preparation | `train_rus_base` (se necessário) |
| `08_modelagem` | Modeling | `model_predictions` + experimentos MLflow |
| `09_avaliacao` | Evaluation | `model_evaluation` + modelo campeão |
| `10_painel_report` | Deployment | Tabelas resumo para dashboard |

O CRISP-DM é **iterativo**: se a avaliação detectar vazamento de informação ou instabilidade temporal, o projeto retorna às etapas de preparação e engenharia de variáveis.

## Estratégia de treino, validação e teste

O **treino e a validação usam o mesmo período** — snapshots de **2024-05-06 a 2024-05-12** (7 dias) — divididos por **sorteio de clientes**: 75% dos clientes para treino e 25% para validação.

| Conjunto | Período | Critério | Proporção |
| --- | --- | --- | --- |
| Treino | 2024-05-06 a 2024-05-12 (7 dias) | Clientes sorteados | 75% dos clientes |
| Validação | 2024-05-06 a 2024-05-12 (mesmo período) | Clientes restantes | 25% dos clientes |
| Teste (out-of-time) | 2024-05-13, 2024-05-14 e 2024-05-15 | Todos os clientes | 100% (holdout temporal) |

**Por que o corte é por cliente e não por linha:** snapshots do mesmo cliente em datas próximas são correlacionados; dividir aleatoriamente por linha colocaria o mesmo cliente em treino e validação no mesmo dia, vazando informação. O sorteio **por cliente** mantém o paralelismo com o uso real (o modelo pontuará clientes ainda não vistos no treino).

**Papel do teste out-of-time (3 safras):** a validação no mesmo período não mede deriva temporal. O OOT usa **três safras distintas de 1 dia** — 2024-05-13, 2024-05-14 e 2024-05-15 — avaliadas de duas formas:

1. **Métrica OOT geral:** todos os snapshots das três safras agrupados, como uma única base de teste;
2. **Métrica por safra:** cada dia avaliado separadamente (AUC, KS, taxa de evento, PSI vs. treino), permitindo observar **tendência/decaimento temporal** dia a dia.

A safra de 2024-05-13 é a mais próxima do treino (menor deriva esperada) e a de 2024-05-15 a mais distante — decaimento monotônico das métricas ao longo das safras é o principal sinal de alerta.

Como o horizonte do target é de apenas 1 dia, **não é necessário gap de purga** entre os períodos (o rótulo de cada snapshot já é conhecido no dia seguinte). Em projetos com horizontes maiores, insira um intervalo de segurança igual ao horizonte.